In [ ]:
import pandas as pd
import numpy as np

train_trans = pd.read_csv("train_transaction.csv")
train_id = pd.read_csv("train_identity.csv")

test_trans = pd.read_csv("test_transaction.csv")
test_id = pd.read_csv("test_identity.csv")

#Merge
train = train_trans.merge(train_id, on="TransactionID", how="left")
test = test_trans.merge(test_id, on="TransactionID", how="left")

#Target values
y = train["isFraud"]
print(y.value_counts(normalize=True))

#Drop missing 
missing = train.isnull().mean()
drop_cols = missing[missing > 0.9].index
common_cols = train.columns.intersection(test.columns)
drop_cols = [col for col in drop_cols if col in common_cols]

train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)


#Missing Values
common_cols = train.columns.intersection(test.columns)
for col in common_cols:
    if train[col].dtype in ['float64','int64']:
        train[col] = train[col].fillna(train[col].median())
        test[col] = test[col].fillna(test[col].median())
    else:
        train[col] = train[col].fillna("missing")
        test[col] = test[col].fillna("missing")


#ُEncoding
from sklearn.preprocessing import LabelEncoder
cat_cols = train.select_dtypes(include='object').columns
common_cat_cols = train.columns.intersection(test.columns)
common_cat_cols = [col for col in common_cat_cols if col in cat_cols]

for col in common_cat_cols:
    
    freq = train[col].value_counts()
    
    train[col] = train[col].map(freq)
    test[col] = test[col].map(freq)
    
    test[col] = test[col].fillna(0)


#Feature Engineering

train["hour"] = (train["TransactionDT"] // 3600) % 24
test["hour"] = (test["TransactionDT"] // 3600) % 24

freq = train["card1"].value_counts()
train["card1_freq"] = train["card1"].map(freq)
test["card1_freq"] = test["card1"].map(freq)

train["DeviceType"] = train["DeviceType"].astype(str)
test["DeviceType"] = test["DeviceType"].astype(str)